# Integrating 28 mouse scRNA-seq samples across 5 tissues and 2 ages

**Data.** 10x Cell Ranger `filtered_feature_bc_matrix.h5` files (mm10), one per
sample, described in `metadata.txt`: five tissues (bone marrow, large
intestine, lung, PBMC, spleen), young (9 weeks) and old (19 months) female SPF
mice. The same mice (ids 1–3, 52–54) were sampled in several tissues.

**What this notebook does**

1. load every sample and attach its metadata;
2. quality control per sample (adaptive MAD thresholds) and doublet removal;
3. normalise, choose highly variable genes across samples, PCA;
4. look at the data **before** integration;
5. integrate with **Harmony**, batch = `sample`;
6. check the integration (sample mixing, marker coherence);
7. cluster, find markers, give clusters provisional broad cell-type labels;
8. cell-type composition by tissue and age;
9. pseudobulk old-vs-young differential expression per tissue × cell type;
10. save the integrated object.

**Two design points to keep in mind**

* Each sample is one mouse at one age, so `sample` is nested inside `age`.
  Correcting for `sample` can also remove part of a real age effect. The
  integrated space is for **clustering and labelling only**; age differences
  in expression are tested on the **raw counts**, summed per mouse
  (pseudobulk), never on the corrected values.
* The replicate for age comparisons is the **mouse** (n = 2–3 per group), not
  the cell. Tests on thousands of cells as if they were independent give
  tiny, meaningless p-values.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import anndata as ad

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

DATA_DIR = Path("data")           # metadata.txt and the <tissue>/<sample>/ folders
OUT_DIR = Path("results/all_tissues")  # everything the notebook writes (not tracked by git)
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 0
DOUBLET_THRESHOLD = 0.25          # scrublet score cut-off (see the doublet section)

sc.settings.verbosity = 1
sc.settings.figdir = OUT_DIR
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(4.5, 4.5))
np.random.seed(SEED)

## 1. Load samples and metadata

The folder for each sample is `<tissue>/<sample>/filtered_feature_bc_matrix.h5`.
Gene symbols are made unique (a few mm10 symbols repeat); Ensembl ids are kept
in `var['gene_ids']`. Raw counts are stored in `layers['counts']` before any
transformation — the doublet detector and the pseudobulk test need them.

## 2. Quality control

Per-cell metrics: total counts, genes detected, and the share of counts in
mitochondrial (`mt-`), ribosomal (`Rps`/`Rpl`) and haemoglobin (`Hba`/`Hbb`)
genes.

**Filtering rule.** Fixed cut-offs do not fit five tissues at once, so
outliers are called **within each sample**: more than 5 median absolute
deviations (MAD) from the sample median on log counts, log genes or the
top-20-gene share, or more than 5 MADs above the median mitochondrial share.
On top of that:

* **mitochondrial ceiling** 25 %, but 50 % for large intestine — intestinal
  epithelium normally has a high mitochondrial share (A109's median is 17 %),
  and a 25 % cap would remove ~40 % of its cells;
* **at least 200 genes** per cell;
* **red blood cells**: more than 50 % haemoglobin reads *and* fewer than 500
  genes. Erythroid precursors in bone marrow and spleen also express
  haemoglobin but detect many more genes, so they are kept.

**Old PBMC samples A170 and A172 are almost entirely red blood cells**
(~90 % of reads from Hbb-bs, Hbb-bt, Hba-a1/2; median ~70 genes per cell) —
most likely red cells were not lysed before loading. Almost nothing is left
after QC, so these samples are dropped (any sample with fewer than
`MIN_CELLS_PER_SAMPLE` cells left is) and **PBMC has young samples only**:
no PBMC age comparison is possible from this data.

### Doublets

Scrublet simulates doublets from each sample's own cells and scores how
doublet-like every real cell is. It runs **per sample** on raw counts, since
doublets only form within one 10x channel.

A fixed threshold (`DOUBLET_THRESHOLD`, 0.25) is used instead of scrublet's
automatic one, which needs scikit-image. Look at
the score histograms: a good threshold sits in the dip between the main
mode and the right tail. To get the automatic threshold instead, run
`pip install scikit-image` and set `threshold=None`.

## 3. Normalise, highly variable genes, PCA

* Library-size normalisation to 10,000 counts and `log1p` (stored in `X`;
  raw counts stay in `layers['counts']`).
* 3,000 highly variable genes with the `seurat_v3` method on raw counts,
  chosen **across samples** (`batch_key="sample"`), so a gene that is variable
  in only one sample does not dominate.
* PCA on those genes (50 components).

## 4. Before integration

If cells from the same tissue split by sample here, there are sample
(batch) effects to correct. Separation **by tissue** is expected for
tissue-specific cells (epithelium, stroma), but shared immune cell types
(B cells, T cells, neutrophils...) found in several tissues should end up
close together after a good integration.

## 5. Harmony

Harmony moves cells in PCA space so that every cluster contains cells from all
samples in proportion. Fast; corrects the PCs only (no corrected counts).

## 6. Check the integration

Two simple checks, before vs after Harmony:

**Sample mixing.** For every cell, look at its 30 nearest neighbours and
count how many come from **other samples of the same tissue**. Compare with
what random mixing would give (the share of the tissue's cells that are not
in the cell's own sample). A ratio near 1 = samples are well mixed; near 0 =
cells only neighbour their own sample. Only same-tissue neighbours count, so
Harmony does not score well simply by blending tissues together.

**Marker coherence.** Over-correction blurs cell types together. The marker
panels below should stay sharp, compact islands after Harmony. If the mixing
ratio is still low, try a larger `theta` (default 2) in `run_harmony`; if
marker islands blur, lower it.

## 7. Clusters, markers and provisional labels

Leiden clustering on the integrated neighbour graph at three resolutions.
`0.5` is used below; finer resolutions are kept for sub-clustering.

### Provisional broad labels

Each cluster is labelled with the broad type whose marker set has the highest
mean score. This is a **starting point, not an annotation**: check every
label against the marker table above and the dot plot, split mixed clusters,
and rename. Tissue-resident types (alveolar macrophages, goblet cells,
AT2 cells...) need finer, tissue-specific markers.

## 8. Composition by tissue and age

Share of each broad type per **sample** (= per mouse and tissue), so each
point is one animal. With 2–3 mice per group these plots show direction and
spread; a formal test needs a compositional method (e.g. `scCODA`, `propeller`)
and should be read cautiously at this sample size.

## 9. Old vs young: pseudobulk differential expression

For every tissue × broad type with at least 2 old and 2 young mice having
≥ 20 cells each, raw counts are **summed per mouse** and tested with
DESeq2 (`pydeseq2`), design `~ age`. This uses the uncorrected counts, so
sample-level correction from the integration cannot erase the age effect —
but neither can it remove a batch effect, because here sample and mouse are
the same thing. Read the hit lists as candidates.

PBMC old samples (A170, A172) differ in depth and cell number from the young
ones; a PBMC age effect may partly be a technical one.

## 10. Save

## Next steps

* Fix the provisional labels by hand, then sub-cluster the big compartments
  (T cells, myeloid, epithelium) per tissue for finer types.
* Re-run section 9 with the refined labels.
* Shared mice across tissues (ids 1–3, 52–54) allow paired analyses, e.g.
  whether a mouse's age shift in bone marrow tracks its shift in blood.